<a href="https://colab.research.google.com/github/Mohommad-Nuzlan-GH/M-R-M-Nuzlan-Database-Analytics-Top-Up-2026/blob/main/2_SQL_in_R_NS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Upload cleaned CSV files into Google Colab
files <- file.choose()

In [ ]:
# Install packages needed for SQL in R
install.packages("sqldf")
install.packages("dplyr")

# Load packages
library(sqldf)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
# Load cleaned datasets into R
CUSTOMER <- read.csv("1#clean_customers.csv")
APP_EVENT <- read.csv("2#clean_app_events.csv")
ORDER_DATA <- read.csv("3#clean_orders.csv")
COMPLAINT <- read.csv("4#clean_complaints.csv")
DELIVERY <- read.csv("5#clean_deliveries.csv")
DRIVER <- read.csv("6#clean_drivers.csv")
VEHICLE <- read.csv("7#clean_vehicles.csv")
HUB <- read.csv("8#clean_hubs.csv")
INCIDENT <- read.csv("9#clean_incidents.csv")

In [ ]:
# Check first rows of each main dataset
head(CUSTOMER)
head(ORDER_DATA)
head(DELIVERY)

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status,age_group
,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,C0001,26,North,Sme,27/11/2024 04:25,44.9,69.2,App,Active,Adult
2,C0002,61,Airport,Consumer,28/10/2025 01:04,55.4,66.6,App,Active,Senior
3,C0003,66,East,Consumer,02/07/2025 03:23,75.9,33.8,Unknown,Active,Senior
4,C0004,75,Central,Consumer,19/08/2025 01:58,32.5,33.0,App,Active,Senior
5,C0005,26,Riverside,Consumer,03/06/2025 06:02,55.9,100.0,Web,Active,Adult
6,C0006,41,West,Consumer,29/03/2024 13:26,39.9,43.3,Web,Active,Middle_Aged


,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag,high_value_order
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>,<chr>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0,True
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,Airport,Low,109.30,App,0,True
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,Airport,High,33.50,Phone,0,False
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,Riverside,North,Medium,10.04,App,1,False
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,South,Low,125.58,Phone,0,True
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,Central,East,High,151.44,Web,1,True


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,delivery_duration_hours,failed_delivery_flag,delayed_delivery_flag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2026-04-28 05:59:54,Failed,17.26,1,0,3.07,12.05,16291.05,1,0
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,,Ontime,10.34,1,0,5.00,13.41,NA,0,0
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,,Ontime,7.92,0,0,4.98,8.51,NA,0,0
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,,Delayed,16.42,0,0,4.18,13.62,NA,0,1
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,,Ontime,14.52,1,0,4.18,9.22,NA,0,0
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2026-04-28 11:52:24,Delayed,13.84,0,0,1.57,9.58,14255.21,0,1


In [ ]:
# Check dataset structure
str(CUSTOMER)
str(ORDER_DATA)
str(DELIVERY)

'data.frame':	650 obs. of  10 variables:
 $ customer_id         : chr  "C0001" "C0002" "C0003" "C0004" ...
 $ age                 : int  26 61 66 75 26 41 54 70 34 23 ...
 $ home_zone           : chr  "North" "Airport" "East" "Central" ...
 $ customer_type       : chr  "Sme" "Consumer" "Consumer" "Consumer" ...
 $ signup_date         : chr  "27/11/2024 04:25" "28/10/2025 01:04" "02/07/2025 03:23" "19/08/2025 01:58" ...
 $ loyalty_score       : num  44.9 55.4 75.9 32.5 55.9 39.9 36.1 84.6 62.6 87.2 ...
 $ app_engagement_score: num  69.2 66.6 33.8 33 100 43.3 39 65.2 40.8 48.6 ...
 $ preferred_channel   : chr  "App" "App" "Unknown" "App" ...
 $ account_status      : chr  "Active" "Active" "Active" "Active" ...
 $ age_group           : chr  "Adult" "Senior" "Senior" "Senior" ...
'data.frame':	1250 obs. of  12 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "P

In [ ]:
#---------------QUESTIONS WE WILL ANSWER--------------------------
# QUERY 1: Overall Delivery Status
# QUERY 2: Hub Performance Analysis
# QUERY 3: Driver Performance Analysis
# QUERY 4: Vehicle Performance Analysis
# QUERY 5: Complaint Analysis
# QUERY 6: Delivery vs Complaints Analysis
# QUERY 7: Incident Analysis
# QUERY 8: App Event Analysis
# QUERY 9: Hub Volume vs Complaints & Failures
# QUERY 10: Factors Linked to Poor Service

In [ ]:
#-------------QUERY 1: Overall Delivery Status---------------------
# 1. Count deliveries by delivery status
query1 <- sqldf("
SELECT
    delivery_status,
    COUNT(*) AS total_deliveries
FROM DELIVERY
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

# 2. Display query result
query1

delivery_status,total_deliveries
<chr>,<int>
Ontime,616
Delayed,202
Failed,132


In [ ]:
#-------------QUERY 2: Hub Performance Analysis---------------------
# 1. Delivery performance by hub
query2 <- sqldf("
SELECT
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating

FROM DELIVERY d
JOIN HUB h
    ON d.hub_id = h.hub_id
GROUP BY h.hub_name, h.zone
ORDER BY delayed_deliveries DESC
")

# 2. Display query result
query2

hub_name,zone,total_deliveries,delayed_deliveries,failed_deliveries,avg_customer_rating
<chr>,<chr>,<int>,<int>,<int>,<dbl>
West Gate,West,127,28,16,3.92
Airport Hub,Airport,104,27,15,3.88
North Exchange,North,136,26,17,3.84
South Link,South,106,26,10,3.95
Central Core,Central,115,25,23,3.68
Riverside Hub,Riverside,115,25,14,3.88
East Dock,East,119,23,11,3.90
Midtown Relay,Central,128,22,26,3.89


In [ ]:
#-------------QUERY 3: Driver Performance Analysis---------------------
# 1. Driver performance analysis
query3 <- sqldf("
SELECT
    dr.driver_id,
    dr.base_zone,
    dr.years_experience,
    dr.training_score,

    COUNT(de.delivery_id) AS total_deliveries,

    SUM(CASE WHEN de.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,

    SUM(CASE WHEN de.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,

    ROUND(AVG(de.customer_rating_post_delivery), 2) AS avg_customer_rating

FROM DELIVERY de

JOIN DRIVER dr
    ON de.driver_id = dr.driver_id

GROUP BY dr.driver_id, dr.base_zone, dr.years_experience, dr.training_score

ORDER BY failed_deliveries DESC, delayed_deliveries DESC

LIMIT 10
")

# 2. Display query result
query3

driver_id,base_zone,years_experience,training_score,total_deliveries,delayed_deliveries,failed_deliveries,avg_customer_rating
<chr>,<chr>,<int>,<dbl>,<int>,<int>,<int>,<dbl>
D133,South,12,88.2,12,2,4,3.42
D024,Riverside,8,71.4,8,0,4,3.44
D104,West,15,87.7,7,0,4,3.93
D055,Central,15,90.5,10,2,3,3.81
D131,South,9,86.7,9,2,3,3.66
D004,Airport,13,88.9,9,1,3,3.51
D083,North,12,80.8,9,1,3,3.91
D010,West,8,70.0,7,0,3,4.15
D092,East,15,88.2,5,0,3,3.38


In [ ]:
#-------------QUERY 4: Vehicle Performance Analysis---------------------
# 1. Vehicle performance analysis
query4 <- sqldf("
SELECT
    v.vehicle_type,
    v.maintenance_status,

    COUNT(d.delivery_id) AS total_deliveries,

    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,

    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,

    ROUND(AVG(v.battery_health_pct), 2) AS avg_battery_health

FROM DELIVERY d

JOIN VEHICLE v
    ON d.vehicle_id = v.vehicle_id

GROUP BY v.vehicle_type, v.maintenance_status

ORDER BY failed_deliveries DESC
")

# 2. Display query result
query4

vehicle_type,maintenance_status,total_deliveries,delayed_deliveries,failed_deliveries,avg_battery_health
<chr>,<chr>,<int>,<int>,<int>,<dbl>
Cargovan,Inrepair,68,16,22,69.82
Hybrid,Inrepair,71,11,21,83.27
Diesel,Inrepair,55,11,17,75.65
Ev,Inrepair,60,14,17,77.79
Hybrid,Active,131,25,15,74.30
Cargovan,Active,117,29,11,74.41
Ev,Active,214,48,11,82.86
Diesel,Active,80,11,8,67.02
Cargovan,Scheduled,38,8,5,68.96


In [ ]:
#-------------QUERY 5: Complaint Analysis---------------------
# 1. Complaint type and severity analysis
query5 <- sqldf("
SELECT
    complaint_type,
    severity,

    COUNT(*) AS total_complaints,

    ROUND(AVG(resolution_days), 2) AS avg_resolution_days,

    ROUND(AVG(compensation_amount), 2) AS avg_compensation

FROM COMPLAINT

GROUP BY complaint_type, severity

ORDER BY total_complaints DESC
")

# 2. Display query result
query5

complaint_type,severity,total_complaints,avg_resolution_days,avg_compensation
<chr>,<chr>,<int>,<dbl>,<dbl>
Delay,Medium,56,5.96,17.23
Missedpickup,Medium,37,6.16,17.43
Driverbehaviour,Medium,31,5.42,15.36
Delay,Low,27,6.48,8.16
Appissue,Medium,25,7.36,15.46
Delay,High,18,12.44,28.42
Driverbehaviour,High,16,13.75,28.79
Missedpickup,High,16,11.56,43.07
Appissue,Low,15,6.07,12.37


In [ ]:
#-------------QUERY 6: Delivery vs Complaints---------------------
# 1. Link delivery status with complaints
query6 <- sqldf("
SELECT
    de.delivery_status,
    COUNT(DISTINCT de.delivery_id) AS total_deliveries,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.resolution_days), 2) AS avg_resolution_days,
    ROUND(AVG(c.compensation_amount), 2) AS avg_compensation

FROM DELIVERY de

LEFT JOIN ORDER_DATA o
    ON de.order_id = o.order_id
LEFT JOIN COMPLAINT c
    ON o.order_id = c.order_id
GROUP BY de.delivery_status
ORDER BY total_complaints DESC
")

# 2. Display query result
query6

delivery_status,total_deliveries,total_complaints,avg_resolution_days,avg_compensation
<chr>,<int>,<int>,<dbl>,<dbl>
Ontime,616,149,7.60,18.53
Delayed,202,48,7.63,16.45
Failed,132,35,9.66,25.47


In [ ]:
#-------------QUERY 7: Incident Analysis---------------------
# 1. Incident analysis by delivery status
query7 <- sqldf("
SELECT
    de.delivery_status,
    i.incident_type,
    i.severity,

    COUNT(i.incident_id) AS total_incidents,

    ROUND(AVG(i.resolved_hours), 2) AS avg_resolution_hours

FROM DELIVERY de

JOIN INCIDENT i
    ON de.delivery_id = i.delivery_id

GROUP BY de.delivery_status, i.incident_type, i.severity

ORDER BY total_incidents DESC
")

# 2. Display query result
query7

delivery_status,incident_type,severity,total_incidents,avg_resolution_hours
<chr>,<chr>,<chr>,<int>,<dbl>
Ontime,Proofmissing,Medium,14,11.11
Ontime,Customernoshow,Low,13,14.35
Ontime,Routedeviation,Low,13,15.48
Ontime,Vehiclefault,Medium,12,4.75
Ontime,Batteryalert,Medium,10,11.02
Ontime,Temperatureissue,Medium,10,15.28
Ontime,Proofmissing,Low,9,14.34
Ontime,Vehiclefault,High,9,13.46
Ontime,Appsyncerror,Low,8,14.67


In [ ]:
#-------------QUERY 8: App Event Analysis---------------------
# 1. App event success and failure analysis
query8 <- sqldf("
SELECT
    event_type,

    COUNT(*) AS total_events,

    SUM(CASE WHEN success_flag = 1 THEN 1 ELSE 0 END) AS successful_events,

    SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) AS failed_events,

    ROUND(AVG(api_latency_ms), 2) AS avg_api_latency

FROM APP_EVENT

GROUP BY event_type

ORDER BY failed_events DESC
")

# 2. Display query result
query8

event_type,total_events,successful_events,failed_events,avg_api_latency
<chr>,<int>,<int>,<int>,<dbl>
payment_retry,69,50,19,472.68
chat_escalated,38,19,19,478.13
track_order,138,138,0,460.71
search_route,99,99,0,456.51
eta_refresh,105,105,0,452.15
delivery_instruction_update,75,75,0,496.29
chat_opened,88,88,0,478.33
cancel_attempt,28,28,0,417.14


In [ ]:
#-------------QUERY 9: Hub Volume vs Complaints & Failures---------------------
# 1. Analyse hub performance with complaints and failures
query9 <- sqldf("
SELECT
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    COUNT(c.complaint_id) AS total_complaints

FROM HUB h

LEFT JOIN DELIVERY d
    ON h.hub_id = d.hub_id
LEFT JOIN ORDER_DATA o
    ON d.order_id = o.order_id
LEFT JOIN COMPLAINT c
    ON o.order_id = c.order_id
GROUP BY h.hub_name, h.zone
ORDER BY total_deliveries DESC
")

# 2. Display result
query9

hub_name,zone,total_deliveries,failed_deliveries,delayed_deliveries,total_complaints
<chr>,<chr>,<int>,<int>,<int>,<int>
North Exchange,North,142,18,26,32
Midtown Relay,Central,132,27,23,35
West Gate,West,129,16,29,28
East Dock,East,121,11,23,33
Riverside Hub,Riverside,118,14,25,33
Central Core,Central,117,23,25,30
South Link,South,108,10,27,18
Airport Hub,Airport,106,15,27,23


In [ ]:
#-------------QUERY 10: Factors Linked to Poor Service---------------------
# 1. Combine multiple factors affecting delivery performance
query10 <- sqldf("
SELECT
    d.delivery_status,
    dr.base_zone,
    dr.training_score,
    v.vehicle_type,
    v.maintenance_status,
    i.incident_type,
    i.severity,

    COUNT(d.delivery_id) AS total_cases,

    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating

FROM DELIVERY d

LEFT JOIN DRIVER dr
    ON d.driver_id = dr.driver_id

LEFT JOIN VEHICLE v
    ON d.vehicle_id = v.vehicle_id

LEFT JOIN INCIDENT i
    ON d.delivery_id = i.delivery_id

GROUP BY
    d.delivery_status,
    dr.base_zone,
    dr.training_score,
    v.vehicle_type,
    v.maintenance_status,
    i.incident_type,
    i.severity

ORDER BY total_cases DESC
")

# 2. Display result
query10

delivery_status,base_zone,training_score,vehicle_type,maintenance_status,incident_type,severity,total_cases,avg_rating
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>
Ontime,Central,69.5,Ev,Active,NA,NA,3,4.14
Ontime,Central,76.1,Ev,Active,NA,NA,3,4.80
Ontime,North,59.6,Hybrid,Active,NA,NA,3,4.50
Ontime,North,90.5,Ev,Active,NA,NA,3,4.15
Ontime,South,64.6,Diesel,Active,NA,NA,3,4.45
Ontime,South,74.1,Ev,Active,NA,NA,3,4.52
Ontime,South,75.2,Cargovan,Active,NA,NA,3,4.26
Ontime,South,84.1,Ev,Active,NA,NA,3,4.00
Ontime,West,74.6,Ev,Active,NA,NA,3,3.84
